In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, RepeatVector, Dense
from sklearn.preprocessing import StandardScaler

from pathlib import Path

dataset_root = Path("LP IV Updated Datasets")
if not dataset_root.is_dir():
    dataset_root = Path("../LP IV Updated Datasets")
print("Dataset folder:", dataset_root.resolve())


In [ ]:
# Load sensor data and keep healthy training examples
df = pd.read_csv(dataset_root / "AutoEncoder/NASA Turbofan ENgine/NASA TurboFan Engine.csv")
print("Loaded data shape:", df.shape)
print("First three rows:")
print(df.head(3))
print("Label counts:")
print(df.Fault_Label.value_counts().sort_index())

features = df.filter(like="sensor").columns
train = df[(df.unit <= 80) & (df.Fault_Label == 0)]
test = df[df.unit > 80]

scaler = StandardScaler()
scaler.fit(train[features])
window_size = 20

# Make sequences of 20 readings from each engine
def create_sequences(data):
    sequences = []
    for engine_id, engine in data.groupby("unit"):
        engine = engine.sort_values("cycle")
        readings = scaler.transform(engine[features])
        for start in range(len(readings) - window_size + 1):
            end = start + window_size
            sequence = readings[start:end]
            sequences.append(sequence)
    return np.array(sequences)

x_train = create_sequences(train)
x_test = create_sequences(test)

print("Training input shape:", x_train.shape)
print("Test input shape:", x_test.shape)
print("Training value range:", x_train.min(), "to", x_train.max())
print("Test value range:", x_test.min(), "to", x_test.max())
print("Selected features:", list(features))
print("Window size:", window_size)
print("First training window: first three readings and five features")
print(x_train[0, :3, :5])
print("Training/test engine counts:", train.unit.nunique(), "/", test.unit.nunique())


In [ ]:
# Create LSTM Autoencoder
model = Sequential([
    Input(shape=(window_size, len(features))),
    LSTM(16),
    RepeatVector(window_size),
    LSTM(16, return_sequences=True),
    Dense(len(features))
])
model.compile(optimizer="adam", loss="mse")
model.summary()


In [ ]:
# Train using normal inputs as their own reconstruction targets
history = model.fit(x_train, x_train, epochs=10, batch_size=128)
print("Last three training epochs:")
print(pd.DataFrame(history.history).tail(3))


In [ ]:
# Reconstruction errors for normal training inputs
train_prediction = model.predict(x_train)
train_error = np.mean((x_train - train_prediction) ** 2, axis=(1, 2))
print("First five training errors:", train_error[:5])
print("Training error min / mean / max:", train_error.min(), train_error.mean(), train_error.max())
threshold = np.percentile(train_error, 95)
print("Anomaly threshold (95th percentile):", threshold)


In [ ]:
# Reconstruction errors for test inputs
test_prediction = model.predict(x_test)
test_error = np.mean((x_test - test_prediction) ** 2, axis=(1, 2))
faults = test_error > threshold
print("First ten test errors:", test_error[:10])
print("Test error min / mean / max:", test_error.min(), test_error.mean(), test_error.max())
print("First ten detected flags:", faults[:10])
print("Faults:", faults.sum())
print("First ten test windows:")
print(pd.DataFrame({
    "Error": test_error[:10],
    "Detected anomaly": faults[:10]
}))
